# ML-08 — Capstone Modeling Lane

This notebook models my **content-refresh prioritisation** lane from W03/W04. The target is `went_dark`: a content item with measured March GSC activity but zero March clicks. The model is a decision-support ranking for pages worth refreshing/reviewing; it is not a causal claim that refreshing will recover traffic.

The Week-4 baseline is the three-signal rule built from February data. This notebook compares a Logistic Regression model against that baseline on the **same February feature universe and the same grouped holdout split**.

## 1. Method choice and why

I choose **Logistic Regression** as the first model because this is a binary classification problem with a small, interpretable feature set. It gives a probability-like risk score, works well as a transparent benchmark, and lets me inspect coefficient direction without rewarding model complexity for its own sake.

The five February decision-time features are the same features established in W03: `feb_impressions`, `feb_clicks`, `feb_ctr`, `feb_position`, and `content_age_days`. The March `went_dark` outcome is the target only.

I will also standardise the numeric features inside a pipeline. No March outcome, label-derived field, or future-window feature is available to the model.

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.inspection import permutation_importance

try:
    from google.colab import userdata
except ImportError:
    userdata = None

hf_token = os.environ.get("HF_TOKEN")
if not hf_token and userdata is not None:
    hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Set HF_TOKEN as a Colab Secret or environment variable before running this notebook.")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute("SET VARIABLE hf_token = ?", [hf_token])
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN getvariable('hf_token'))")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIM = f"{REL}/dim_content.parquet"
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Warehouse connection ready. Decision window: February 2026; target window: March 2026.")

## 2. Split design

I use a **grouped holdout by `client_hash_id`**: 70% of clients are used for training and 30% of clients are held out for evaluation. This is more honest than a random row split because multiple pages from the same client can have similar site-level behaviour. It tests whether the model generalises to unseen clients rather than memorising client-specific patterns.

The split is created once with `random_state=42`, and both the Logistic Regression model and the Week-4 baseline are evaluated on exactly the same held-out rows.

In [ ]:
# Recreate the W03 labelled decision universe.
# IMPORTANT: March fields are used only to define y; none enter X.
model_df = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS feb_impressions,
        SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS feb_clicks,
        SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE)
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE), 0) AS feb_position
    FROM {FEB}
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) >= 100
       AND SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) >= 3
),
march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) FILTER (WHERE gsc_data_available IS TRUE) AS march_clicks,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS march_measured_days
    FROM {MAR}
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.feb_impressions,
    f.feb_clicks,
    f.feb_clicks / NULLIF(f.feb_impressions, 0) AS feb_ctr,
    f.feb_position,
    DATE_DIFF('day', d.content_created_date, DATE '2026-02-28') AS content_age_days,
    COALESCE(m.march_clicks, 0) AS march_clicks,
    COALESCE(m.march_measured_days, 0) AS march_measured_days
FROM feb f
JOIN {DIM} d USING (client_hash_id, content_hash_id)
LEFT JOIN march m USING (client_hash_id, content_hash_id)
WHERE d.is_published IS TRUE
  AND d.content_created_date <= DATE '2026-02-28'
""").df()

model_df = model_df[model_df["march_measured_days"] > 0].copy()
model_df["went_dark"] = (model_df["march_clicks"] == 0).astype(int)

feature_cols = ["feb_impressions", "feb_clicks", "feb_ctr", "feb_position", "content_age_days"]

# Deterministic grouped split.
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, test_idx = next(gss.split(model_df[feature_cols], model_df["went_dark"], groups=model_df["client_hash_id"]))
train = model_df.iloc[train_idx].copy()
test = model_df.iloc[test_idx].copy()

print(f"Rows: {len(model_df):,} | train: {len(train):,} | test: {len(test):,}")
print(f"Clients: {model_df.client_hash_id.nunique():,} | train clients: {train.client_hash_id.nunique():,} | test clients: {test.client_hash_id.nunique():,}")
print(f"Train went_dark rate: {train.went_dark.mean():.1%} | Test went_dark rate: {test.went_dark.mean():.1%}")
assert set(train.client_hash_id).isdisjoint(set(test.client_hash_id))
print("Client-group leakage check: PASS")

## 3. Train + compare vs my baseline

The Week-4 baseline is reconstructed exactly from the same rule: +2 for age ≥365 days, +2 for CTR <1%, and +1 for position ≥10. For an apples-to-apples binary comparison, `REFRESH_REVIEW` (baseline score ≥4) is treated as the baseline's positive prediction for `went_dark`.

The Logistic Regression model is evaluated at a 0.50 probability threshold for the same binary target. The main comparison uses **F1**, because `went_dark` is an action-prioritisation target and accuracy alone can hide poor performance on the positive class. ROC-AUC is also reported as a threshold-independent ranking metric.

In [ ]:
X_train = train[feature_cols]
y_train = train["went_dark"]
X_test = test[feature_cols]
y_test = test["went_dark"]

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])
model.fit(X_train, y_train)

model_prob = model.predict_proba(X_test)[:, 1]
model_pred = (model_prob >= 0.50).astype(int)

# Week-4 baseline, using only February inputs.
baseline_score = (
    (X_test["content_age_days"] >= 365).astype(int) * 2
    + (X_test["feb_ctr"] < 0.01).astype(int) * 2
    + (X_test["feb_position"] >= 10).astype(int)
)
baseline_pred = (baseline_score >= 4).astype(int)


def metrics(name, y_true, pred, prob):
    return {
        "method": name,
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, prob),
    }

comparison = pd.DataFrame([
    metrics("Week-4 baseline", y_test, baseline_pred, baseline_score),
    metrics("Logistic Regression", y_test, model_pred, model_prob),
])

comparison = comparison.round(3)
print(comparison.to_string(index=False))

print("\nConfusion matrix — baseline")
print(confusion_matrix(y_test, baseline_pred))
print("\nConfusion matrix — Logistic Regression")
print(confusion_matrix(y_test, model_pred))

In [ ]:
# Inspect the model's learned directions and permutation importance on the held-out clients.
coef = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": model.named_steps["logreg"].coef_[0]
}).sort_values("coefficient", ascending=False)
print("Standardised Logistic Regression coefficients:")
print(coef.to_string(index=False))

perm = permutation_importance(
    model, X_test, y_test,
    scoring="f1", n_repeats=10, random_state=42
)
importance = pd.DataFrame({
    "feature": feature_cols,
    "mean_f1_drop": perm.importances_mean,
    "std": perm.importances_std
}).sort_values("mean_f1_drop", ascending=False)
print("\nPermutation importance (mean F1 drop):")
print(importance.round(4).to_string(index=False))

## 4. Errors and interpretation

The error review separates false positives from false negatives. A false positive is a page the model prioritises as `went_dark` risk even though it received March clicks; a false negative is a page that went dark but the model missed.

For operational use, the most important question is whether false positives are acceptable review work and whether false negatives represent pages that should have been prioritised. The model should only replace the baseline if its improvement is meaningful on the held-out clients, not merely because it is more sophisticated.

In [ ]:
errors = test[["client_hash_id", "content_hash_id"] + feature_cols + ["went_dark"]].copy()
errors["model_prob"] = model_prob
errors["pred"] = model_pred
errors["error_type"] = np.select(
    [(errors.went_dark == 0) & (errors.pred == 1), (errors.went_dark == 1) & (errors.pred == 0)],
    ["false_positive", "false_negative"],
    default="correct"
)

fp = errors[errors.error_type == "false_positive"].sort_values("model_prob", ascending=False)
fn = errors[errors.error_type == "false_negative"].sort_values("model_prob", ascending=False)

print(f"False positives: {len(fp):,}")
print(fp[["content_hash_id", "model_prob", "feb_ctr", "feb_position", "content_age_days"]].head(10).round(3).to_string(index=False))
print(f"\nFalse negatives: {len(fn):,}")
print(fn[["content_hash_id", "model_prob", "feb_ctr", "feb_position", "content_age_days"]].head(10).round(3).to_string(index=False))

# Compare model vs baseline row-by-row on the same held-out set.
error_compare = pd.DataFrame({
    "baseline_correct": baseline_pred == y_test.to_numpy(),
    "model_correct": model_pred == y_test.to_numpy(),
})
print("\nSame-test-set outcomes:")
print(error_compare.value_counts().rename("n"))

### Interpretation to complete after execution

- If Logistic Regression has a materially higher F1 and ROC-AUC than the baseline, it is evidence that combining the five signals continuously can improve the hand-written thresholds. That is still directional evidence, not proof of causal lift.
- If the baseline matches or beats the model, the simple rule remains the better benchmark. Do not keep the model merely because it is ML.
- Coefficients and permutation importance show which February signals the model actually uses. If `content_age_days` or `feb_ctr` dominates, that should be checked against the Week-4 signal audit rather than treated as surprising proof.
- False positives are not automatically bad: for a review queue, a small amount of extra manual review may be acceptable. False negatives matter because they are pages the model fails to prioritise despite the observed March outcome.

**Decision rule:** prefer the model only if its held-out-client F1 improvement is meaningful and the error profile is operationally acceptable. Otherwise retain the Week-4 baseline and use the model as an exploratory signal.

## 5. Self-check

- [x] Method choice is justified for the binary content-refresh lane.
- [x] Split is grouped by client, with the same held-out rows used for baseline and model.
- [x] Baseline and model use the same five decision-time features and the same target.
- [x] F1, precision, recall, accuracy, and ROC-AUC are reported.
- [x] Coefficients and permutation importance interpret the model.
- [x] False positives and false negatives are inspected.
- [x] No March outcome field is used as an input feature.
- [ ] Run all cells with `HF_TOKEN`, inspect the actual model-vs-baseline numbers, complete the interpretation with the observed result, and commit the executed notebook to `work/notebooks/w05_model.ipynb`.